In [13]:
import os
import re
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [14]:
df = pd.read_csv("../data/slm_eval_dataset.csv")

print("Dataset shape:", df.shape)
print(df["expected_risk"].value_counts())

df.head()

Dataset shape: (20, 2)
expected_risk
Low       7
High      7
Medium    6
Name: count, dtype: int64


,input,expected_risk
0,License: MIT License Family: Permissive Projec...,Low
1,License: MIT License Family: Permissive Projec...,Low
2,License: Apache-2.0 License Family: Permissive...,Low
3,License: BSD-3-Clause License Family: Permissi...,Low
4,License: LGPL-2.1 License Family: Weak Copylef...,Medium


In [15]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully")
print("CUDA available:", torch.cuda.is_available())

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully
CUDA available: True


In [16]:
def build_prompt(scenario):
    return f"""
You are a strict OSS license compliance risk classifier.

Choose exactly one risk label using these rules:

Low = permissive licenses such as MIT, BSD, Apache, CC0, Unlicense.
Medium = weak copyleft licenses such as LGPL, MPL, EPL, CDDL.
High = strong copyleft, proprietary, unknown, or custom licenses such as GPL, AGPL, Proprietary, Unknown, Custom.

Scenario:
{scenario}

Return exactly:
Risk: <Low|Medium|High>
Reason: <one short sentence>
"""

In [17]:
def predict_risk(scenario):
    prompt = build_prompt(scenario)

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip()

In [18]:
def extract_risk(response):
    for line in response.split("\n"):
        line = line.strip().lower()

        if line.startswith("risk:"):
            value = line.replace("risk:", "").strip()

            if value.startswith("high"):
                return "High"
            elif value.startswith("medium"):
                return "Medium"
            elif value.startswith("low"):
                return "Low"

    return "Unknown"

In [19]:
responses = []
predicted_risks = []

for idx, row in df.iterrows():
    print(f"Processing {idx + 1}/{len(df)}")

    response = predict_risk(row["input"])
    predicted_risk = extract_risk(response)

    responses.append(response)
    predicted_risks.append(predicted_risk)

df["qwen_response"] = responses
df["predicted_risk"] = predicted_risks

df.head()

Processing 1/20
Processing 2/20
Processing 3/20
Processing 4/20
Processing 5/20
Processing 6/20
Processing 7/20
Processing 8/20
Processing 9/20
Processing 10/20
Processing 11/20
Processing 12/20
Processing 13/20
Processing 14/20
Processing 15/20
Processing 16/20
Processing 17/20
Processing 18/20
Processing 19/20
Processing 20/20


,input,expected_risk,qwen_response,predicted_risk
0,License: MIT License Family: Permissive Projec...,Low,Risk: Low\nReason: The MIT License is a permis...,Low
1,License: MIT License Family: Permissive Projec...,Low,Risk: Low\nReason: The MIT License is a permis...,Low
2,License: Apache-2.0 License Family: Permissive...,Low,Risk: Low\nReason: The Apache-2.0 license is a...,Low
3,License: BSD-3-Clause License Family: Permissi...,Low,Risk: Low\nReason: The BSD-3-Clause License is...,Low
4,License: LGPL-2.1 License Family: Weak Copylef...,Medium,Risk: Medium\nReason: The LGPL-2.1 is a weak c...,Medium


In [20]:
y_true = df["expected_risk"]
y_pred = df["predicted_risk"]

accuracy = accuracy_score(y_true, y_pred)
report = classification_report(y_true, y_pred, zero_division=0)
cm = confusion_matrix(y_true, y_pred)

print("Qwen Base SLM Accuracy:", accuracy)

print("\nClassification Report:")
print(report)

print("\nConfusion Matrix:")
print(cm)

Qwen Base SLM Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

        High       1.00      1.00      1.00         7
         Low       1.00      1.00      1.00         7
      Medium       1.00      1.00      1.00         6

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20


Confusion Matrix:
[[7 0 0]
 [0 7 0]
 [0 0 6]]


In [21]:
os.makedirs("../outputs", exist_ok=True)

df.to_csv("../outputs/step4_qwen_base_predictions.csv", index=False)

with open("../outputs/step4_qwen_base_report.txt", "w") as f:
    f.write("Step 4: Qwen Base SLM Evaluation\n\n")
    f.write(f"Model: {model_name}\n")
    f.write(f"Dataset: slm_eval_dataset.csv\n")
    f.write(f"Total samples: {len(df)}\n")
    f.write(f"Accuracy: {accuracy}\n\n")
    f.write("Classification Report:\n")
    f.write(report)
    f.write("\nConfusion Matrix:\n")
    f.write(str(cm))

print("Step 4 outputs saved successfully.")

Step 4 outputs saved successfully.
